# Enfoque Truncado — Regresión Regularizada (Lasso / Ridge)

Este cuaderno visualiza los resultados del pipeline `32_modelo_truncado.py`.

**Objetivo:** Predecir el **semestre exacto de término** de estudiantes de generaciones ≤ 2016,
eliminando el sesgo de censura al trabajar únicamente con trayectorias completas.

| Elemento | Detalle |
|----------|---------|
| N | 3,157 observaciones |
| Predictores | 40 (S1-S8: reg, esf, nat, real + interacciones) |
| Variable objetivo | `semestre_termino` (continua, 8–20) |
| Modelos | Ridge (α óptimo por 5-CV) y Lasso (α óptimo por 5-CV) |
| IC | Bootstrapping no paramétrico, B = 2000 |
| H. pruebas | F-test global, t-test por coeficiente, Diebold-Mariano |

In [ ]:
import json
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings('ignore')

BASE_DIR  = Path('../../').resolve()
IMG_DIR   = BASE_DIR / 'imagenes'
JSON_PATH = Path('metricas_modelo_truncado.json')

sns.set_theme(style='whitegrid', context='notebook', font_scale=1.2)

with open(JSON_PATH, 'r', encoding='utf-8') as f:
    m = json.load(f)

print('✅ Métricas cargadas')
print(f"  N = {m['n_obs']} | p = {m['n_features']} | "
      f"ȳ = {m['y_mean']:.2f} ± {m['y_std']:.2f} semestres")

---
## 1. Desempeño Global — RMSE, MAE y R² con IC al 95 %

In [ ]:
rows = []
for mod in ['ridge', 'lasso']:
    bs = m[mod]['bootstrap']
    ft = m[mod]['f_test']
    rows.append({
        'Modelo': f"{mod.capitalize()} (α={m[mod]['alpha']:.4f})",
        'RMSE': f"{bs['rmse_mean']:.4f}",
        'IC 95% RMSE': f"[{bs['rmse_ci_lo']:.4f}, {bs['rmse_ci_hi']:.4f}]",
        'MAE': f"{bs['mae_mean']:.4f}",
        'IC 95% MAE': f"[{bs['mae_ci_lo']:.4f}, {bs['mae_ci_hi']:.4f}]",
        'R²': f"{bs['r2_mean']:.4f}",
        'IC 95% R²': f"[{bs['r2_ci_lo']:.4f}, {bs['r2_ci_hi']:.4f}]",
        'F (global)': f"{ft['F']:.2f}",
        'p (F-test)': f"{ft['p_value']:.2e}",
        'Activos': m[mod]['n_active'],
    })

df_sum = pd.DataFrame(rows).set_index('Modelo')
print('=== Tabla Resumen ===')
display(df_sum)

In [ ]:
img = plt.imread(IMG_DIR / 'trunc_comparativa_ic.png')
fig, ax = plt.subplots(figsize=(16, 5))
ax.imshow(img); ax.axis('off')
plt.tight_layout(); plt.show()

---
## 2. Predicción vs. Real — Ridge y Lasso

In [ ]:
img = plt.imread(IMG_DIR / 'trunc_pred_vs_real.png')
fig, ax = plt.subplots(figsize=(16, 7))
ax.imshow(img); ax.axis('off')
plt.tight_layout(); plt.show()

---
## 3. Gráficas de Residuos y Prueba de Normalidad (Shapiro-Wilk)

In [ ]:
img = plt.imread(IMG_DIR / 'trunc_residuos.png')
fig, ax = plt.subplots(figsize=(16, 6))
ax.imshow(img); ax.axis('off')
plt.tight_layout(); plt.show()

---
## 4. Importancia de Variables

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
for ax, fname, tag in [
    (axes[0], 'trunc_ridge_importancia.png', 'Ridge'),
    (axes[1], 'trunc_lasso_importancia.png', 'Lasso'),
]:
    img = plt.imread(IMG_DIR / fname)
    ax.imshow(img); ax.axis('off')
    ax.set_title(tag, fontsize=12, pad=10)
plt.tight_layout(); plt.show()

In [ ]:
# Top 10 Ridge vs Lasso en tabla
for mod in ['ridge', 'lasso']:
    imp = pd.Series(m[mod]['importances']).sort_values(ascending=False).head(10)
    df_imp = pd.DataFrame({'Variable': imp.index,
                           '|Coef| std': imp.values.round(4)}).reset_index(drop=True)
    print(f'\n=== Top 10 — {mod.capitalize()} ===')
    display(df_imp)

---
## 5. Prueba t sobre Coeficientes Individuales

Se reportan los coeficientes con su error estándar asintótico, estadístico t y p-valor.
Coeficientes Lasso en cero se omiten (selección por penalización).

In [ ]:
for mod in ['ridge', 'lasso']:
    df_t = pd.DataFrame(m[mod]['coef_table'])
    sig = df_t[df_t['Significativo'] == '✓']
    print(f'\n=== Coeficientes Significativos (p < 0.05) — {mod.capitalize()} ===')
    display(sig.reset_index(drop=True))
    print(f'  ({len(sig)} de {len(df_t)} coeficientes activos son significativos)')

---
## 6. Prueba de Hipótesis: F-test Global y Diebold-Mariano

In [ ]:
dm = m['diebold_mariano']

print('='*60)
print('  F-TEST GLOBAL (H0: todos los coeficientes = 0)')
print('='*60)
for mod in ['ridge', 'lasso']:
    ft = m[mod]['f_test']
    print(f"\n  {mod.capitalize()}: F({ft['df1']}, {ft['df2']}) = {ft['F']:.2f}, "
          f"p = {ft['p_value']:.2e}")
    print(f"  → {ft['decision']}")

print()
print('='*60)
print('  DIEBOLD-MARIANO (H0: Lasso = Ridge en error cuadrático)')
print('='*60)
print(f"  DM = {dm['DM_statistic']:.4f},  p = {dm['p_value']:.4f}")
print(f"  d̄ (diferencia media errores) = {dm['d_bar']:.6f}")
print(f"  → {dm['decision']}")

---
## 7. Síntesis de Hallazgos

| Dimensión | Ridge | Lasso |
|-----------|-------|-------|
| α óptimo (5-CV) | 7.7426 | 0.0024 |
| RMSE (IC 95%) | 1.360 [1.307, 1.421] | 1.360 [1.307, 1.421] |
| MAE (IC 95%) | 0.957 [0.924, 0.991] | 0.957 [0.925, 0.992] |
| R² (IC 95%) | 0.755 [0.738, 0.772] | 0.755 [0.738, 0.772] |
| Coefs activos | 40/40 | 31/40 |
| F-test global | F=240.05, p≈0 | F=240.03, p≈0 |
| Diebold-Mariano | — | DM=0.026, p=0.979 |

> **Conclusión principal:** Con R² ≈ 0.755, el modelo explica el **75.5 %** de la varianza del
> semestre de término con un error absoluto medio de **< 1 semestre**.
> Lasso selecciona 31 de 40 predictores sin pérdida de desempeño (DM test no significativo).
> El predictor dominante en ambos modelos es `s8_reg` (regularidad en S8), con un coeficiente
> negativo altamente significativo: cada unidad de regularidad acumulada **reduce** el semestre
> de término en ~2.4–2.6 semestres.